# Module 06 — Lesson 02
# Finding and Removing Duplicates

---

> The HR department just merged exports from two different payroll systems into one file. Some employees show up twice because both systems had them. Some show up twice with *different* salaries, because one system was updated more recently than the other. And a couple show up under names typed differently — `"Sanjay Singh"` in one system, `"SANJAY SINGH"` in the other. Deduplication sounds like it should be one line of code. It isn't — because the real question is never "are these rows identical," it's "which row is *correct*."

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Detect exact duplicate rows with `.duplicated()` and remove them with `.drop_duplicates()`
- Detect **business-key duplicates** — rows that repeat on an identifying column (like an ID) even though other columns differ — using `subset=`
- Decide which duplicate to keep using `keep='first'`, `keep='last'`, or an explicit sort beforehand
- Catch **near-duplicates** caused by inconsistent text formatting (case, whitespace) that `.duplicated()` misses by default
- Recognize when two identical-looking rows are *not* duplicates at all, and shouldn't be merged

---
## Why Duplicates Aren't Just Annoying — They're Misleading

A duplicate row doesn't just waste space. It quietly double-counts. If you're computing average salary by department and one employee's row appears twice, that person's salary gets weighted twice as heavily as everyone else's — the average shifts, and nothing in the output tells you it happened.

Duplicates show up in real data for a handful of very common reasons, and each one implies a different fix:

| Cause | What it looks like | Fix |
|---|---|---|
| A script or import ran twice | Fully identical rows | Drop the exact copy |
| Two source systems both export the same record | Same ID, possibly different values | Keep the more trustworthy/recent one |
| The same record was re-submitted after an update | Same ID, one column changed (e.g. a raise) | Keep the *latest* by date, not just any one |
| Inconsistent manual data entry | Same real-world entity, different text formatting | Normalize text, *then* check for duplicates |

Let's work through all four using the merged export.

In [1]:
import pandas as pd

employees = pd.read_csv("data/employees_raw.csv")
print(f"employees.shape: {employees.shape}")
employees.head(10)

employees.shape: (61, 8)


,employee_id,name,department,age,years_experience,salary,performance_rating,submitted_date
0,1059,Priya Gupta,Engineering,32.0,4.1,78700.0,3.0,2024-01-02
1,1021,Vikram Pillai,HR,51.0,1.2,51100.0,3.0,2024-01-05
2,1015,SANJAY SINGH,Sales,46.0,1.8,57200.0,4.0,2024-02-06
3,1016,Neha Reddy,Engineering,27.0,3.0,70700.0,4.0,2024-01-01
4,1047,Simran Patel,Engineering,27.0,1.6,69800.0,4.0,2024-01-25
5,1027,Arjun Das,Marketing,38.0,4.6,62100.0,4.0,2024-01-12
6,1008,Tanvi Pillai,Engineering,37.0,2.7,79300.0,4.0,2024-02-03
7,1034,Preeti Patel,Engineering,57.0,11.3,103500.0,4.0,2024-01-20
8,1015,Sanjay Singh,Sales,46.0,1.8,57200.0,4.0,2024-02-06
9,1042,Vikram Mehta,HR,40.0,4.2,54400.0,5.0,2024-02-09


---
## 1. Exact Duplicate Rows

`.duplicated()` flags every row that is a byte-for-byte match of an earlier row (`True` on the second and later copies, `False` on the first). `.drop_duplicates()` removes them, keeping the first occurrence by default.

In [2]:
exact_dupe_mask = employees.duplicated()
print(f"Exact duplicate rows: {exact_dupe_mask.sum()}")
employees[exact_dupe_mask].sort_values("employee_id")

Exact duplicate rows: 4


,employee_id,name,department,age,years_experience,salary,performance_rating,submitted_date
34,1002,Riya Singh,Engineering,43.0,10.0,95300.0,5.0,2024-01-26
16,1008,Tanvi Pillai,Engineering,37.0,2.7,79300.0,4.0,2024-02-03
59,1028,Rahul Sharma,Engineering,34.0,5.7,80900.0,4.0,2024-01-29
23,1059,Priya Gupta,Engineering,32.0,4.1,78700.0,3.0,2024-01-02


In [3]:
employees_no_exact_dupes = employees.drop_duplicates()
print(f"{employees.shape[0]} -> {employees_no_exact_dupes.shape[0]} rows "
      f"({employees.shape[0] - employees_no_exact_dupes.shape[0]} exact duplicates removed)")

61 -> 57 rows (4 exact duplicates removed)


---
## 2. Business-Key Duplicates: Same ID, Different Values

Removing exact duplicates isn't the end of the story. Check for duplicate `employee_id`s now — you'll find more rows flagged than exact duplicates alone caught, because some employees were re-submitted with an **updated salary**.

In [4]:
id_dupe_mask = employees_no_exact_dupes.duplicated(subset=["employee_id"], keep=False)
print(f"Rows sharing an employee_id with another row: {id_dupe_mask.sum()}")

(employees_no_exact_dupes[id_dupe_mask]
    .sort_values(["employee_id", "submitted_date"])
    [["employee_id", "name", "salary", "submitted_date"]])

Rows sharing an employee_id with another row: 16


,employee_id,name,salary,submitted_date
2,1015,SANJAY SINGH,57200.0,2024-02-06
8,1015,Sanjay Singh,57200.0,2024-02-06
3,1016,Neha Reddy,70700.0,2024-01-01
47,1016,Neha Reddy,73500.0,2024-01-06
44,1023,Rohan Chauhan,52400.0,2024-02-02
58,1023,Rohan Chauhan,54400.0,2024-02-07
48,1028,Rahul Sharma,80900.0,2024-01-29
17,1028,Rahul Sharma,84100.0,2024-02-27
10,1030,Suresh Bose,82600.0,2024-01-11
27,1030,Suresh Bose,82600.0,2024-01-11


Look at `employee_id` 1016, 1023, 1028, 1034, and 1047: same person, same everything else, but a *different salary on a later date*. That's not noise to drop — it's a raise. The **later submission is the correct one**, so keeping `keep='first'` blindly would silently throw away the up-to-date salary and keep the stale one.

The fix: sort by date first, then drop duplicates keeping the **last** (most recent) row per `employee_id`.

In [5]:
employees_deduped = (
    employees_no_exact_dupes
    .sort_values("submitted_date")
    .drop_duplicates(subset=["employee_id"], keep="last")
    .sort_values("employee_id")
    .reset_index(drop=True)
)

print(f"{employees_no_exact_dupes.shape[0]} -> {employees_deduped.shape[0]} rows after business-key dedup")

# Confirm employee 1034 kept the LATER, higher salary
employees_deduped[employees_deduped["employee_id"] == 1034]

57 -> 49 rows after business-key dedup


,employee_id,name,department,age,years_experience,salary,performance_rating,submitted_date
28,1034,Preeti Patel,Engineering,57.0,11.3,110900.0,4.0,2024-02-06


---
## 3. Near-Duplicates: Same Person, Inconsistent Text

`.duplicated()` only catches values that match *exactly*. It's still possible for the same employee to slip through as two different `employee_id`... except here they share an ID but the row wasn't flagged as identical because the `name` field was typed differently. Compare employee 1015 before and after normalizing case and whitespace.

In [6]:
raw_1015 = employees_no_exact_dupes[employees_no_exact_dupes["employee_id"] == 1015]
print("Before normalizing -- names look different even though they're the same person:")
print(raw_1015["name"].tolist())

normalized_names = raw_1015["name"].str.strip().str.title()
print("\nAfter .str.strip().str.title():")
print(normalized_names.tolist())

Before normalizing -- names look different even though they're the same person:
['SANJAY SINGH', 'Sanjay Singh']

After .str.strip().str.title():
['Sanjay Singh', 'Sanjay Singh']


In [7]:
# Apply the same normalization to the whole dataset BEFORE the final dedup step,
# so text formatting differences can't hide a real duplicate
employees_deduped["name"] = employees_deduped["name"].str.strip().str.title()
employees_deduped.head(10)

,employee_id,name,department,age,years_experience,salary,performance_rating,submitted_date
0,1001,Rohan Pillai,HR,27.0,6.4,56900.0,4.0,2024-02-07
1,1002,Riya Singh,Engineering,43.0,10.0,95300.0,5.0,2024-01-26
2,1003,Karan Kapoor,Engineering,25.0,2.3,74400.0,3.0,2024-01-28
3,1004,Rohan Bose,Marketing,46.0,13.6,85200.0,5.0,2024-02-05
4,1005,Kabir Verma,HR,45.0,0.7,50700.0,NaN,2024-01-24
5,1007,Suresh Pillai,Engineering,50.0,1.5,76200.0,5.0,2024-02-01
6,1008,Tanvi Pillai,Engineering,37.0,2.7,79300.0,4.0,2024-02-03
7,1009,Pooja Gupta,Sales,51.0,9.7,79600.0,4.0,2024-01-10
8,1010,Simran Das,Engineering,23.0,0.5,70600.0,NaN,2024-01-03
9,1012,Sneha Bhat,Engineering,40.0,5.4,81800.0,4.0,2024-01-13


---
## 4. Not Every Look-Alike Is a Duplicate

It's tempting to treat any two rows with the same `name` as the same person. Don't — two genuinely different employees can share a common name. The only column you can trust as a true identity key here is `employee_id`. Check: does this dataset have any *same-name, different-ID* pairs that a name-only dedup would have wrongly merged?

In [8]:
same_name_mask = employees_deduped.duplicated(subset=["name"], keep=False)
same_name_rows = employees_deduped[same_name_mask].sort_values("name")

if same_name_rows.empty:
    print("No same-name collisions in this dataset -- but always check on real data before")
    print("assuming 'name' is a safe column to deduplicate on.")
else:
    print("These rows share a name but are DIFFERENT people (different employee_id):")
    print(same_name_rows[["employee_id", "name", "department"]])

No same-name collisions in this dataset -- but always check on real data before
assuming 'name' is a safe column to deduplicate on.


---
## ⚠️ Common Mistakes

In [9]:
# -- Mistake 1: Calling drop_duplicates() with no subset and assuming that's enough --------

raw = pd.read_csv("data/employees_raw.csv")
only_exact = raw.drop_duplicates()

print("Mistake 1 — default drop_duplicates() only catches FULLY identical rows.")
print(f"  Rows after default drop_duplicates(): {only_exact.shape[0]}")
print(f"  Employee IDs that still repeat: {only_exact.duplicated(subset=['employee_id']).sum()}")
print("  -> Business-key duplicates (same ID, updated values) need subset= explicitly.")

Mistake 1 — default drop_duplicates() only catches FULLY identical rows.
  Rows after default drop_duplicates(): 57
  Employee IDs that still repeat: 8
  -> Business-key duplicates (same ID, updated values) need subset= explicitly.


In [10]:
# -- Mistake 2: Using the default keep='first' when the LATER row is the correct one -------

wrong_order = raw.drop_duplicates(subset=["employee_id"], keep="first")
salary_1034_wrong = wrong_order.loc[wrong_order["employee_id"] == 1034, "salary"].iloc[0]

right_order = raw.sort_values("submitted_date").drop_duplicates(subset=["employee_id"], keep="last")
salary_1034_right = right_order.loc[right_order["employee_id"] == 1034, "salary"].iloc[0]

print("Mistake 2 — keep='first' on unsorted data keeps whichever row happened to load")
print("first, not the most accurate one:")
print(f"  Salary kept with default keep='first' (no sort): {salary_1034_wrong:,.0f}")
print(f"  Salary kept after sorting by date, keep='last':   {salary_1034_right:,.0f}")
print("  -> Always ask 'which row is correct', not just 'which row is first'.")

Mistake 2 — keep='first' on unsorted data keeps whichever row happened to load
first, not the most accurate one:
  Salary kept with default keep='first' (no sort): 103,500
  Salary kept after sorting by date, keep='last':   110,900
  -> Always ask 'which row is correct', not just 'which row is first'.


In [11]:
# -- Mistake 3: Deduplicating on a human-entered text column without normalizing it first ---

messy_name_dupe_count = raw.duplicated(subset=["employee_id", "name"]).sum()

normalized = raw.copy()
normalized["name"] = normalized["name"].str.strip().str.title()
clean_name_dupe_count = normalized.duplicated(subset=["employee_id", "name"]).sum()

print("Mistake 3 — checking duplicates on raw text under-counts them, because")
print("'Sanjay Singh' and 'SANJAY SINGH' don't match as strings:")
print(f"  Duplicates found using raw 'name' text: {messy_name_dupe_count}")
print(f"  Duplicates found after .str.strip().str.title(): {clean_name_dupe_count}")
print("  -> Normalize text columns BEFORE running duplicate checks that depend on them.")

Mistake 3 — checking duplicates on raw text under-counts them, because
'Sanjay Singh' and 'SANJAY SINGH' don't match as strings:
  Duplicates found using raw 'name' text: 9
  Duplicates found after .str.strip().str.title(): 12
  -> Normalize text columns BEFORE running duplicate checks that depend on them.


---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Count Every Kind of Duplicate

Load `data/employees_raw.csv` fresh. Print: the number of exact duplicate rows, and the number of rows that share an `employee_id` with another row (use `keep=False` so both copies count).

In [12]:
# Your code here

### Exercise 2 — Drop Exact Duplicates Only

Remove just the fully-identical rows with `.drop_duplicates()` (no `subset`). Print the shape before and after.

In [13]:
# Your code here

### Exercise 3 — Keep the Most Recent Submission

Starting from your Exercise 2 result, sort by `submitted_date` and use `drop_duplicates(subset=['employee_id'], keep='last')` to resolve the remaining business-key duplicates. Confirm employee `1023` ends up with their higher, more recent salary.

In [14]:
# Your code here

### Exercise 4 — Normalize Before You Check

On your Exercise 3 result, normalize the `name` column with `.str.strip().str.title()`, then confirm employee `1015` now appears with one consistent name spelling.

In [15]:
# Your code here

### Exercise 5 — (Challenge) Write a Reusable Dedup Function

Write a function `deduplicate_employees(df)` that applies, in order: exact-duplicate removal, text normalization on `name`, and business-key deduplication keeping the latest `submitted_date`. Run it on the raw data and print the final shape.

In [16]:
# Your code here

---
## 📌 Key Takeaways

- **`.duplicated()` only catches exact row matches by default** — real-world duplicates usually repeat on a business key (like an ID) while other columns differ, which needs `subset=` to catch.
- **When duplicates disagree, decide which row is correct — don't default to `keep='first'` blindly** — sort by a timestamp (or other trust signal) first, so `keep='last'` (or `'first'`) actually means something.
- **Normalize text (case, whitespace) before checking for duplicates on it** — otherwise the same real-world entity can hide behind two differently formatted strings and never get flagged.